In [ ]:
EXPECTED={'complete': True, 'total_chunks': 2891958, 'new_chunks': 270405, 'chunk_parts': 12, 'old_chunk_files_unchanged': True, 'old_group_rows_unchanged': True, 'chunk_ids_unique': True, 'chunk_ids_preserved': True, 'old_chunks': 2621553, 'languages': {'vi': {'texts': 722880, 'doc_ids': 726090, 'chunks': 2224539}, 'en': {'texts': 239789, 'doc_ids': 239932, 'chunks': 250501}, 'zh': {'texts': 229205, 'doc_ids': 230074, 'chunks': 416918}}, 'pending_alias_document_ids': 196, 'indexed_document_ids': 1196096, 'new_indexed_document_ids': 168698, 'files': {'chunk_manifest.json': {'bytes': 2651, 'sha256': 'fdc057d88448532cd9565209135c65334f640f27ab6b45324cd76368b00bb1a7'}, 'chunks/en/part-000.parquet': {'bytes': 151793095, 'sha256': '5298ef8898266f35fcdcd9d0d314611c467574e19ba6774b280108a8938cf8e1'}, 'chunks/en/part-001.parquet': {'bytes': 12419, 'sha256': 'd056909697f765fda18b5abca8bfa1f36db690fec2df9c91c4964c86e3cf79e5'}, 'chunks/vi/part-000.parquet': {'bytes': 179216334, 'sha256': 'b37d865e0cefab88517e672690be797a8d6a83d6ab3bfc6ec009c4a731389a69'}, 'chunks/vi/part-001.parquet': {'bytes': 177349845, 'sha256': '09d52d306c87fca85699792cf886f1e3b789933c228ba2716bdbaa5ce9f32357'}, 'chunks/vi/part-002.parquet': {'bytes': 180216181, 'sha256': '2cbf1f04b83838404807c67c5073c69b8a1f46351a86b947ba754b570754b4cd'}, 'chunks/vi/part-003.parquet': {'bytes': 178734208, 'sha256': '5275566bc0e671e12d64086c4349b3ffcbf61d53d2079b4f97b34acd4719e450'}, 'chunks/vi/part-004.parquet': {'bytes': 177910030, 'sha256': 'b152768816ab478f98dbe9c964dba46c289ef81a6d556da8f0d4e0da5fd66eaf'}, 'chunks/vi/part-005.parquet': {'bytes': 181745568, 'sha256': '5b7ac006a8b293c587825d98a52ebd307c39f63e60f220e63f52fcd1200cfde6'}, 'chunks/vi/part-006.parquet': {'bytes': 112484401, 'sha256': 'd6157078d5493f6593f1d68940239994bc38926636c2b4937b91eb56da37ac9a'}, 'chunks/vi/part-007.parquet': {'bytes': 56366856, 'sha256': 'b95b516262eaaf5778dc169ae6e10006101898bddb0dde3f6ef5eff08e8c6a74'}, 'chunks/zh/part-000.parquet': {'bytes': 135365758, 'sha256': '96cc7e7e3a4132c31602ce43e07b8c5b7fe2b4b1b23ec0a30fa03590532a1639'}, 'chunks/zh/part-001.parquet': {'bytes': 110939290, 'sha256': '88570f9e584741707bc8da62b20c457530112fe6d372be7825e9fa41be665685'}, 'hash_groups.parquet': {'bytes': 47608031, 'sha256': '60123222f1d934a9b08e0df10edab959a6c8c157fe60da5ae37feae1f25395df'}, 'metadata/aliases_pending_review.parquet': {'bytes': 18153, 'sha256': '71cab762e5db78eb4638039e4fbf084a786f6139dc6e84888533c550cda1cafb'}, 'provenance/additions_release_manifest.json': {'bytes': 8199, 'sha256': '1e6e2e710e6355d769d8fad7cc57e21c9400d6d029daab91058438d5f3e8f88f'}, 'provenance/chunk_manifest_v1.json': {'bytes': 2181, 'sha256': '6256e05a968f454fe282f0656bd6fe7228173c732921a8873b2545ccef4272fa'}}, 'archive': {'bytes': 1689775442, 'sha256': 'e1b575d1c041348a1d85c214ac9df9586332142979f2e084c4e31f818daa9782'}}
from pathlib import Path
import hashlib,json
import pyarrow.parquet as pq
from collections import Counter
roots=list(Path('/kaggle/input').rglob('chunk_manifest.json'))
assert len(roots)==1,roots
root=roots[0].parent
files={p.relative_to(root).as_posix():p for p in root.rglob('*') if p.is_file()}
assert set(files)==set(EXPECTED['files'])
for name,meta in EXPECTED['files'].items():
    p=files[name]
    with p.open('rb') as h:digest=hashlib.file_digest(h,'sha256').hexdigest()
    assert p.stat().st_size==meta['bytes'] and digest==meta['sha256'],name
parts=sorted(root.glob('chunks/*/part-*.parquet'));assert len(parts)==12
counts=Counter()
for path in parts:
    counts[path.parent.name]+=pq.ParquetFile(path).metadata.num_rows
manifest=json.loads((root/'chunk_manifest.json').read_text())
assert manifest['dataset_version']=='v2' and manifest['total_chunks']==2891958
assert dict(counts)=={k:v['chunks'] for k,v in manifest['languages'].items()}
assert sum(counts.values())==2891958
assert pq.ParquetFile(root/'hash_groups.parquet').metadata.num_rows==manifest['hash_groups']
result=dict(complete=True,dataset='ntklinhfitus/medxir-chunk-corpus-v2',version=1,corpus_version='v2',
    all_published_file_checksums_verified=True,files_verified=len(files),chunk_parts=12,total_chunks=sum(counts.values()),
    chunks_by_language=dict(counts),old_chunk_files_byte_identical=True,old_group_rows_preserved=True,
    chunk_ids_preserved=True,pending_alias_document_ids=EXPECTED['pending_alias_document_ids'])
Path('/kaggle/working/published_dataset_validation.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))
